# Evidencia de los otros nueve temas de la guía
Ejercicios propios para los proyectos 1 a 6 y 10 a 12 definidos en el PDF del profesor. Utilizan Gapminder y se pueden repetir sin conexión. El proyecto 1 extrae una tabla desde una página HTML local; no ejecuta una descarga desde Internet. Estos ejercicios no son copias ejecutadas de los notebooks originales del autor.

In [1]:
from pathlib import Path
from io import StringIO
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
ruta = Path('gapminder_full.csv')
if not ruta.exists(): ruta = Path('/content/gapminder_full.csv')
if not ruta.exists(): raise FileNotFoundError('Sube gapminder_full.csv')
df = pd.read_csv(ruta)
print('Dataset:', df.shape, 'países:', df.country.nunique())

Dataset: (1704, 6) países: 142


## Proyecto 1  Extracción de una tabla HTML local
Probamos la extracción de una tabla con pandas.read_html. Para una web real habría que consultar una página pública y respetar sus condiciones.

In [2]:
html = df.loc[(df.year == 2007) & df.country.isin(['Mexico','Brazil','Argentina']), ['country','life_exp','gdp_cap']].to_html(index=False)
extraida = pd.read_html(StringIO(html))[0]
print(extraida.to_string(index=False))
assert extraida.shape == (3,3)

  country  life_exp      gdp_cap
Argentina    75.320 12779.379640
   Brazil    72.390  9065.800825
   Mexico    76.195 11977.574960


## Proyecto 2  Limpieza
Insertamos deliberadamente un nulo y una fila duplicada en una copia pequeña y luego los corregimos.

In [3]:
prueba = pd.concat([df.head(8), df.head(1)], ignore_index=True)
prueba.loc[1,'life_exp'] = np.nan
antes = (int(prueba.life_exp.isna().sum()), int(prueba.duplicated().sum()))
limpio = prueba.drop_duplicates().copy()
limpio['life_exp'] = limpio.life_exp.fillna(limpio.life_exp.median())
print('Antes, nulos y duplicados:', antes, 'después:', (int(limpio.life_exp.isna().sum()), int(limpio.duplicated().sum())))
assert len(limpio) == 8 and limpio.life_exp.notna().all()

Antes, nulos y duplicados: (1, 1) después: (0, 0)


## Proyecto 3  Filtros lógicos, loc e iloc

In [4]:
filtrado = df.loc[(df.year == 2007) & df.country.isin(['Mexico','Brazil']), ['country','year','life_exp']]
print(filtrado.to_string(index=False))
print('Primera por posición:', filtrado.iloc[0].to_dict())
assert len(filtrado) == 2

country  year  life_exp
 Brazil  2007    72.390
 Mexico  2007    76.195
Primera por posición: {'country': 'Brazil', 'year': 2007, 'life_exp': 72.39}


## Proyecto 4  Groupby y agg
Resumimos por continente las observaciones de 2007.

In [5]:
agrupado = df[df.year == 2007].groupby('continent').agg(promedio_vida=('life_exp','mean'), mediana_vida=('life_exp','median'), paises=('country','nunique')).round(2)
print(agrupado.to_string())
assert agrupado.shape[0] == 5

           promedio_vida  mediana_vida  paises
continent                                     
Africa             54.81         52.93      52
Americas           73.61         72.90      25
Asia               70.73         72.40      33
Europe             77.65         78.61      30
Oceania            80.72         80.72       2


## Proyecto 5  Pivot table
Obtenemos una matriz de años por continentes.

In [6]:
pivote = pd.pivot_table(df,index='year',columns='continent',values='life_exp',aggfunc='mean')
print('Forma:', pivote.shape)
print(pivote.round(1).tail(3).to_string())
assert pivote.shape == (12,5)

Forma: (12, 5)
continent  Africa  Americas  Asia  Europe  Oceania
year                                              
1997         53.6      71.2  68.0    75.5     78.2
2002         53.3      72.4  69.2    76.7     79.7
2007         54.8      73.6  70.7    77.6     80.7


## Proyecto 6  Concat y merge
Reconstruimos dos periodos y unimos dos tablas por claves.

In [7]:
reunido = pd.concat([df[df.year <= 1977],df[df.year > 1977]],ignore_index=True)
unido = df[['country','year','life_exp']].merge(df[['country','year','gdp_cap']],on=['country','year'],validate='one_to_one')
print('Concat:', reunido.shape, 'merge:', unido.shape)
assert len(reunido) == len(unido) == len(df)

Concat: (1704, 6) merge: (1704, 4)


## Proyecto 10  Modelo base
Clasificamos esperanza de vida alta frente a baja. La esperanza de vida define el objetivo y no entra entre los predictores.

In [8]:
y = (df.life_exp >= df.life_exp.median()).astype(int)
X = df[['year','population','gdp_cap','continent']]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
pre = ColumnTransformer([
  ('nums',Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())]),['year','population','gdp_cap']),
  ('cat',OneHotEncoder(handle_unknown='ignore'),['continent'])])
base = Pipeline([('pre',pre),('tree',DecisionTreeClassifier(max_depth=3,random_state=42))])
base.fit(X_train,y_train)
base_acc = accuracy_score(y_test,base.predict(X_test))
print('Entrenamiento:',len(X_train),'prueba:',len(X_test),'exactitud base:',round(base_acc,3))

Entrenamiento: 1363 prueba: 341 exactitud base: 0.877


## Proyecto 11  Búsqueda de parámetros
GridSearchCV examina nueve combinaciones con validación cruzada de tres pliegues, solo sobre entrenamiento.

In [9]:
search = GridSearchCV(base,{'tree__max_depth':[2,3,5],'tree__min_samples_leaf':[1,5,15]},cv=3,scoring='accuracy',n_jobs=1)
search.fit(X_train,y_train)
print('Mejores parámetros:',search.best_params_,'exactitud CV:',round(search.best_score_,3))

Mejores parámetros: {'tree__max_depth': 5, 'tree__min_samples_leaf': 15} exactitud CV: 0.891


## Proyecto 12  Evaluación
La matriz indica aciertos y errores sobre datos de prueba. Como el mismo país puede tener varios años en ambos conjuntos, esta cifra no estima rendimiento en países nunca vistos.

In [10]:
pred = search.predict(X_test)
matrix = confusion_matrix(y_test,pred)
final_acc = accuracy_score(y_test,pred)
print('Exactitud prueba:',round(final_acc,3),'matriz:',matrix.tolist())
print(classification_report(y_test,pred,target_names=['Baja','Alta']))
assert matrix.sum() == len(X_test)

Exactitud prueba: 0.88 matriz: [[149, 22], [19, 151]]
              precision    recall  f1-score   support

        Baja       0.89      0.87      0.88       171
        Alta       0.87      0.89      0.88       170

    accuracy                           0.88       341
   macro avg       0.88      0.88      0.88       341
weighted avg       0.88      0.88      0.88       341

